# DEPRECATED OmniVoice migration reference
**Not registered. Not selectable. Not production.** This notebook is intentionally blocked and remains only until the Azure TTS to OpenVoice compatibility gate passes or an explicit cleanup removes it. Use the OpenVoice Colab or Kaggle notebook instead.

In [ ]:
raise RuntimeError('DEPRECATED: OmniVoice is not registered, not selectable, and not production')

In [ ]:
%pip install --no-cache-dir -q 'omnivoice==0.2.1' 'google-cloud-storage>=2.16'
import importlib.metadata
print({'omnivoice': importlib.metadata.version('omnivoice')})

In [ ]:
from google.colab import auth
auth.authenticate_user()
print('Google authentication complete')

In [ ]:
PROJECT_ID='bakgethwa'
BUCKET='bakgethwa-mathula-tv'
PREFIX='mathula-tv'
JOB_ID='19ba6d69f1b84132ba4f20599101834a'
CALIBRATION_ONLY=True
FORCE=False
CALIBRATION_APPROVED=False
LEASE_SECONDS=900
print({'project':PROJECT_ID,'bucket':BUCKET,'prefix':PREFIX,'job_id':JOB_ID,'calibration_only':CALIBRATION_ONLY,'force':FORCE})

In [ ]:
import os
from google.colab import userdata
HF_TOKEN=userdata.get('HUGGINGFACE_TOKEN')
if not HF_TOKEN: raise RuntimeError('Add HUGGINGFACE_TOKEN in Colab Secrets and grant notebook access')
os.environ['HF_TOKEN']=HF_TOKEN
print('Hugging Face secret is available')

In [ ]:
import json, pathlib, tempfile
from google.cloud import storage
client=storage.Client(project=PROJECT_ID); bucket=client.bucket(BUCKET)
assert bucket.exists(), f'Cannot access gs://{BUCKET}'
manifest=json.loads(bucket.blob(f'{PREFIX}/runtime/manifest.json').download_as_text())
print({'package_version':manifest['package_version'],'wheel_object':manifest['wheel_object'],'wheel_size':manifest['wheel_size']})

In [ ]:
import hashlib, subprocess, sys
wheel=pathlib.Path(tempfile.gettempdir())/pathlib.Path(manifest['wheel_object']).name
bucket.blob(manifest['wheel_object']).download_to_filename(str(wheel))
assert wheel.stat().st_size==manifest['wheel_size'], 'Wheel size mismatch'
assert hashlib.sha256(wheel.read_bytes()).hexdigest()==manifest['sha256'], 'Wheel SHA-256 mismatch'
subprocess.run([sys.executable,'-m','pip','install','--quiet','--force-reinstall','--no-deps','--no-cache-dir',str(wheel)],check=True)
print({'wheel_verified':True,'installed':wheel.name})

In [ ]:
from mathula_tv.omnivoice_worker import discover_synthesis_jobs
eligible=discover_synthesis_jobs(client,BUCKET,PREFIX)
status_blob=bucket.blob(f'{PREFIX}/jobs/{JOB_ID}/status.json')
assert status_blob.exists(), 'Configured job does not exist'
if not CALIBRATION_ONLY: assert JOB_ID in eligible or FORCE, f'Configured job is not synthesis-eligible: {eligible}'
print({'eligible_jobs':eligible,'selected_job':JOB_ID,'calibration_only':CALIBRATION_ONLY})

In [ ]:
from mathula_tv.gcs_store import GCSStore
from mathula_tv.omnivoice_worker import OmniVoiceSynthesisWorker
store=GCSStore(BUCKET,PREFIX,client=client); local_root=pathlib.Path(tempfile.gettempdir())/'mathula-tv-omnivoice'; job_dir=local_root/JOB_ID
if CALIBRATION_ONLY:
    for relative in ('input/source_audio.wav','analysis/transcript_en.json','analysis/pyannote_diarization.json','translation/transcript_zu.json'): store.download(JOB_ID,relative,job_dir/relative)
    references={}
else:
    worker=OmniVoiceSynthesisWorker(store,JOB_ID,local_root,force=FORCE,lease_seconds=LEASE_SECONDS)
    try: references=worker.prepare()
    except Exception as exc: worker.fail(exc); raise
print({'lease':'not-required-for-calibration' if CALIBRATION_ONLY else 'claimed','inputs_downloaded':True})

In [ ]:
print({'reference_selection':'calibration helper will select the same authoritative speaker reference' if CALIBRATION_ONLY else {speaker:{'start':r['start'],'end':r['end'],'duration':r['duration'],'text_words':len(r['transcript'].split()),'sha256':r['sha256']} for speaker,r in references.items()}})

In [ ]:
from mathula_tv.omnivoice_adapter import RealOmniVoiceAdapter
adapter=RealOmniVoiceAdapter(target_locale='zu-ZA')
if CALIBRATION_ONLY: model_info=adapter.load()
else:
    worker.adapter=adapter
    try: model_info=worker.load_model()
    except Exception as exc: worker.fail(exc); raise
print({**model_info,'language_id':adapter.language_id})

In [ ]:
from mathula_tv.omnivoice_worker import calibrate_local_job
from mathula_tv.media import checksum
if CALIBRATION_ONLY:
    calibration=calibrate_local_job(job_dir,adapter); directory=job_dir/'output/calibration'
    for name in ('current_settings.wav','space_matched_settings.wav','comparison_manifest.json'):
        path=directory/name; relative=f'output/calibration/{name}'; temp=f'{relative}.partial.calibration'
        store.upload(JOB_ID,temp,path,if_generation_match=0); store.promote(JOB_ID,temp,relative,checksum(path),checksum)
else:
    try: calibration=worker.calibrate()
    except Exception as exc: worker.fail(exc); raise
print(json.dumps(calibration,indent=2))
print('Download and listen to both calibration WAVs before approving full synthesis')

In [ ]:
assert CALIBRATION_APPROVED, 'Stop here. Listen to current_settings.wav and space_matched_settings.wav, then set CALIBRATION_APPROVED=True.'
print('Calibration approved; full synthesis enabled')

In [ ]:
try: segment_results=worker.synthesize_turns()
except Exception as exc: worker.fail(exc); raise
print({'synthesized_segments':len(segment_results),'generation_seconds':round(sum(x['generation_time'] for x in segment_results),2)})

In [ ]:
try: report=worker.assemble_and_verify()
except Exception as exc: worker.fail(exc); raise
print({'final_duration':report['final_duration'],'overlaps':report['overlaps'],'duration_mismatches':report['duration_mismatches']})

In [ ]:
quality=report['loudness_measurements']
assert quality['valid'] and quality['channels']==1 and quality['sample_rate']==24000
assert report['output_sha256']
print({'quality_valid':True,'non_silent_ratio':quality['non_silent_ratio'],'clipped_ratio':quality['clipped_ratio']})

In [ ]:
try: objects=worker.upload_results()
except Exception as exc: worker.fail(exc); raise
print({'dubbed_audio':objects['dubbed_audio'],'processing_report':objects['processing_report'],'remote_verified':True})

In [ ]:
try: final_status=worker.complete()
except Exception as exc: worker.fail(exc); raise
print({'job_id':JOB_ID,'state':final_status['state'],'stage':'omnivoice_synthesis','segments':len(segment_results),'speakers':report['speaker_count'],'model':report['model_id'],'next':'Run the server process command'})